In [ ]:
import sys
import gc
import json
from pathlib import Path

import torch

# Robust repo-root import (works in local + Colab)
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root / "Efficient_Architecture" / "src").exists():
    repo_root = repo_root / "Efficient_Architecture"
sys.path.append(str(repo_root))

from src.utils.models import Qwen3, Qwen35, LFM2, IBM_Granite1b, IBM_Granite, GPT2
from data.preprocessing import c4_dataset
from src.utils.metrics import (
    profiling_test_suite,
    profile_prefill_decode,
    non_embedding_params,
    count_params,
)

# Configuration
LANGUAGE = "en"
SPLIT = "train"
NUM_EXAMPLES = 1  # keep profiling lightweight

OUT_DIR = Path("profiling_metrics")
TRACE_DIR = Path("profiling_traces")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)

model_classes = {
    "Qwen3": Qwen3,
    "Qwen3.5": Qwen35,
    "LFM2": LFM2,
    "IBM-G1B": IBM_Granite1b,
    "IBM-G350M": IBM_Granite,
    "GPT2": GPT2,
}

for name, ModelClass in model_classes.items():
    print(f"\n=== Profiling {name} ===")

    wrapper = None
    model = None
    result = {"_model": name, "_status": "ok", "_points": {}}

    try:
        wrapper = ModelClass()
        model = wrapper.model

        # Parameter counts are cheap; helpful for correlating capacity vs performance.
        total_p = count_params(model)
        non_emb_p = non_embedding_params(model)
        result["_params"] = {
            "total": total_p,
            "non_embedding": non_emb_p,
            "embedding_plus_head": total_p - non_emb_p,
        }

        # Try to move to GPU for GPU profiling. If it OOMs, we continue on CPU.
        if torch.cuda.is_available():
            try:
                model = model.to("cuda")
            except Exception as e:
                print(f"Could not move {name} to CUDA; profiling on CPU. Error: {e}")
                model = wrapper.model  # keep whatever device it was on

        for (read_len, gen_len) in profiling_test_suite:
            key = f"({read_len},{gen_len})"
            print(f"  - {key}")

            try:
                dataset = c4_dataset(split=SPLIT, language=LANGUAGE, tokenizer=wrapper.tokenizer)
                dataloader = dataset.process(num_examples=NUM_EXAMPLES, sequence_length=read_len, batch_size=1)
                batch = next(iter(dataloader))

                traces_subdir = TRACE_DIR / name
                trace_prefix = f"r{read_len}_g{gen_len}"

                point_result = profile_prefill_decode(
                    model,
                    batch,
                    decode_steps=gen_len,
                    traces_dir=str(traces_subdir),
                    trace_prefix=trace_prefix,
                )
                result["_points"][key] = point_result

            except torch.cuda.OutOfMemoryError as e:
                result["_points"][key] = {"_status": "oom", "error": str(e)}
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                msg = str(e).lower()
                if "out of memory" in msg:
                    result["_points"][key] = {"_status": "oom", "error": str(e)}
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                else:
                    result["_points"][key] = {"_status": "error", "error": str(e)}
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    except Exception as e:
        result["_status"] = "error"
        result["_error"] = str(e)
        print(f"Failed to profile {name}: {e}")

    # Save per-model profiling JSON regardless of success
    out_path = OUT_DIR / f"{name}_profiling.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Saved {out_path}")

    # Free model memory before next model
    del model
    del wrapper
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
